In [1]:
import os

# Paste your API token from the Kaggle popup here
os.environ["KAGGLE_API_TOKEN"] = "KGAT_2b013b5dcbafb2b8d60c189cfe8aa563"

print("Kaggle API Token configured!")

Kaggle API Token configured!


In [2]:
import kagglehub
import os

# 2. Download the dataset
print("Downloading JMuBEN dataset...")
dataset_path = kagglehub.dataset_download("noamaanabdulazeem/jmuben-coffee-dataset")

print(f"Dataset downloaded successfully to: {dataset_path}")

# 3. Verify the contents to see the folder structure
print("\nFolders inside the dataset:")
print(os.listdir(dataset_path))

Using Colab cache for faster access to the 'jmuben-coffee-dataset' dataset.
Dataset downloaded successfully to: /kaggle/input/jmuben-coffee-dataset

Folders inside the dataset:
['JMuBEN']


In [3]:
import tensorflow as tf
import matplotlib.pyplot as plt

# 1. Define your parameters
batch_size = 16
img_height = 224 # Required input size for MobileNetV3
img_width = 224

# 2. Only load the classes relevant to my Capstone scope
target_classes = ['Healthy', 'Leaf rust', 'Miner'] # Corrected capitalization for 'Leaf rust' again

# Adjust dataset_path to point to the correct subdirectory
import os
full_dataset_path = os.path.join(dataset_path, 'JMuBEN')

print("Loading Training Data...")
print(f"Target classes being used: {target_classes}") # Added print statement to verify target_classes
train_ds = tf.keras.utils.image_dataset_from_directory(
  full_dataset_path, # This is the corrected variable
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  class_names=target_classes # This automatically filters out Phoma and Cerscospora!
)

print("\nLoading Validation Data...")
print(f"Target classes being used: {target_classes}") # Added print statement to verify target_classes
val_ds = tf.keras.utils.image_dataset_from_directory(
  full_dataset_path,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size,
  class_names=target_classes
)

# 3. Optimize for performance (Crucial for large datasets on Colab)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("\nData Pipeline Ready!")

Loading Training Data...
Target classes being used: ['Healthy', 'Leaf rust', 'Miner']
Found 44297 files belonging to 3 classes.
Using 35438 files for training.

Loading Validation Data...
Target classes being used: ['Healthy', 'Leaf rust', 'Miner']
Found 44297 files belonging to 3 classes.
Using 8859 files for validation.

Data Pipeline Ready!


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# ---------------------------------------------------------
# VISUALIZATION 1: 3x3 Sample Image Grid
# ---------------------------------------------------------
print("Generating Image Sample Grid...")
plt.figure(figsize=(10, 10))
# Fix: Use target_classes directly as train_ds loses class_names after caching/prefetching
class_names = target_classes

# Track how many images of each class we've plotted
plotted_counts = {name: 0 for name in class_names}

# Scan through the dataset to find exactly 3 of each class
for images, labels in train_ds.unbatch().take(500):
    class_name = class_names[labels.numpy()]

    if plotted_counts[class_name] < 3:
        # Calculate grid position: Row = class index, Col = current count
        row = class_names.index(class_name)
        col = plotted_counts[class_name]
        position = (row * 3) + col + 1

        ax = plt.subplot(3, 3, position)
        plt.imshow(images.numpy().astype("uint8"))
        plt.title(f"{class_name}", fontsize=14)
        plt.axis("off")

        plotted_counts[class_name] += 1

    # Stop searching once we have 3 of each
    if all(count == 3 for count in plotted_counts.values()):
        break

plt.tight_layout()
plt.show()

Generating Image Sample Grid...


In [ ]:
# ---------------------------------------------------------
# VISUALIZATION 2: Class Distribution Bar Chart
# ---------------------------------------------------------
print("\nGenerating Class Distribution Chart...")

class_counts = []
for class_name in target_classes:
    # Correct the path: dataset_path points to the root,
    # but images are in JMuBEN/class_name
    folder_path = os.path.join(full_dataset_path, class_name)
    count = len(os.listdir(folder_path))
    class_counts.append(count)

plt.figure(figsize=(8, 5))
bars = plt.bar(target_classes, class_counts, color=['#2ca02c', '#d62728', '#1f77b4'])

# Add the exact numbers on top of the bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 50, int(yval), va='bottom', ha='center', fontsize=12)

plt.title('Image Distribution for AgroInsight Target Classes', fontsize=14)
plt.xlabel('Coffee Leaf Condition', fontsize=12)
plt.ylabel('Number of Images', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# ---------------------------------------------------------
# MODEL ARCHITECTURE: MobileNetV3 (Small)
# ---------------------------------------------------------
print("Building MobileNetV3 Transfer Learning Model...")

# 1. Load the pre-trained base model (without the top classification layer)
base_model = tf.keras.applications.MobileNetV3Small(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Freeze the base model to prevent destroying pre-trained weights during initial training
base_model.trainable = False

# 3. Add the custom classification head for AgroInsight
model = tf.keras.Sequential([
    # Add data augmentation to prevent overfitting
    tf.keras.layers.RandomFlip('horizontal_and_vertical', input_shape=(img_height, img_width, 3)),
    tf.keras.layers.RandomRotation(0.2),

    # Base Model
    base_model,

    # Flatten the features using Global Average Pooling (more efficient than standard Flatten)
    tf.keras.layers.GlobalAveragePooling2D(),

    # Dense hidden layer with Dropout
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2), # Drops 20% of neurons to prevent memorization

    # Output layer (3 neurons for Healthy, Leaf rust, Miner)
    tf.keras.layers.Dense(3, activation='softmax')
])

# 4. Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display the architecture
model.summary()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# ---------------------------------------------------------
# INITIAL PERFORMANCE METRICS
# ---------------------------------------------------------
epochs = 5 # Short run for MVP demonstration

print(f"Training Model for {epochs} epochs...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

# Extract predictions for the validation set
print("\nGenerating Classification Report and Confusion Matrix...")
y_pred = []
y_true = []

# Iterate through validation data to get true labels and predict
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

# 1. Print Accuracy, Precision, Recall, F1-Score
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=target_classes))

# 2. Plot Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_classes,
            yticklabels=target_classes)
plt.title('Validation Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Diagnosis', fontsize=12)
plt.ylabel('Actual Diagnosis', fontsize=12)
plt.show()

# 3. Plot Training vs Validation Accuracy
plt.figure(figsize=(8, 4))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Initial Training Accuracy over 5 Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ---------------------------------------------------------
# DEPLOYMENT OPTION: API Mockup (FastAPI / Swagger UI)
# ---------------------------------------------------------
# We use %%writefile to save this script as app.py.
# In a real deployment (e.g., on a cloud server or local testing),
# running `uvicorn app:app --reload` will automatically generate a Swagger UI at /docs

%%writefile app.py
from fastapi import FastAPI, File, UploadFile
from pydantic import BaseModel
import uvicorn

app = FastAPI(
    title="AgroInsight Edge-AI API",
    description="API for classifying Rwandan Arabica coffee leaf diseases (Healthy, Leaf rust, Miner).",
    version="1.0.0"
)

# Mock response class
class DiagnosisResult(BaseModel):
    filename: str
    disease_class: str
    confidence_score: float

@app.post("/predict/", response_model=DiagnosisResult, tags=["Diagnosis Endpoint"])
async def predict_disease(file: UploadFile = File(...)):
    """
    Upload an image of a coffee leaf (224x224) to receive a diagnostic prediction.
    *This endpoint automatically generates a Swagger UI interface.*
    """

    # In full production, the image loading and model.predict() logic goes here.
    # For MVP demonstration, we return a mock successful prediction.

    mock_prediction = "Leaf rust"
    mock_confidence = 0.94

    return {
        "filename": file.filename,
        "disease_class": mock_prediction,
        "confidence_score": mock_confidence
    }

@app.get("/", tags=["Health Check"])
def health_check():
    return {"status": "AgroInsight Inference Engine is Active"}

# To view the Swagger UI locally:
# 1. Install dependencies: pip install fastapi uvicorn python-multipart
# 2. Run the server: uvicorn app:app --reload
# 3. Open browser to: http://127.0.0.1:8000/docs